In [0]:
%sql
-- Definir o contexto do Unity Catalog
USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS retail_analytics;
USE SCHEMA retail_analytics;

In [0]:
import os
from pyspark.sql import functions as F

# Caminho relativo para a pasta data/raw dentro do Git Folder
repo_path = os.getcwd().rsplit('/notebooks', 1)[0]
raw_path = f"{repo_path}/data/raw"

# 1. Leitura do ficheiro JSON de produtos
df_products = spark.read \
    .option("multiline", "true") \
    .json(f"file:{raw_path}/products.json")

# 2. Adicionar metadados de ingestão (boa prática Delta Lake / Bronze)
df_products_bronze = df_products \
    .withColumn("_ingestion_timestamp", F.current_timestamp()) \
    .withColumn("_source_file", F.lit("products.json"))

# 3. Escrita no Unity Catalog como tabela Delta
df_products_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.retail_analytics.bronze_products")

print("[INFO] Tabela 'bronze_products' criada com sucesso no Unity Catalog!")